In [1]:
import pandas as pd
import yfinance as yf
import time
import os
import re
import sys
from IPython.display import clear_output
import gspread

# Load arguments

In [2]:
GENERATE_MARKDOWN = True
GENERATE_HTML = True
REPORT_PATH = '../TradingAssistWebapp/pages/'
GSHEET_CREDS = "c:/users/pbara/Documents/Python/secrets/sheets-pandas-reader-193e91a08e8e.json"

In [3]:
#ADJ is the amount in dollars to make up for the price diff bw/ yfinance api and broker data
yfinance_sym_dic = { 
    'MNQ': {'SYM':'MNQ=F', 'ADJ': 0},
    'NQ': {'SYM':'NQ=F', 'ADJ': 0},
    'MES': {'SYM':'MES=F', 'ADJ': 0},
    'ES': {'SYM':'ES=F', 'ADJ': 0},
    'US100': {'SYM':'MNQ=F', 'ADJ': -53.18},
    'GC': {'SYM':'GC=F', 'ADJ': 0},
    'MGX': {'SYM':'MGC=F', 'ADJ': 0},
    'SI': {'SYM':'SI=F', 'ADJ': 0},
    'SIL': {'SYM':'SIL=F', 'ADJ': 0},
    'XAUUSD': {'SYM':'GC=F', 'ADJ': 0},
    'AGXUSD': {'SYM':'SI=F', 'ADJ': 0},
    'BZ': {'SYM':'BZ=F', 'ADJ': 0}, # Brent Crude Futures
    'CL': {'SYM':'CL=F', 'ADJ': 0}, # WTI Crude Futures
    'BTC': {'SYM':'BTC-USD', 'ADJ': 0},
    'ETH': {'SYM':'ETH-USD', 'ADJ': 0},
    'ETH.i': {'SYM':'ETH-USD', 'ADJ': 0}
}


def get_live_price(ticker_symbol: str, yfinance_map: dict)-> float:
    # Initialize the Ticker object 
    if ticker_symbol in yfinance_map.keys():
        ticker = yf.Ticker(yfinance_map[ticker_symbol]['SYM'])
        # .fast_info provides the most recent 'last_price'
        # This is faster than fetching the full .info dictionary
        current_price = ticker.fast_info['last_price'] + yfinance_map[ticker_symbol]['ADJ']
    else:
        ticker = yf.Ticker(ticker_symbol)
        current_price = ticker.fast_info['last_price']
        
    return current_price

# Read the trades worksheet

In [4]:
# Replace with your actual Google Sheet ID
# (Found in the URL: https://docs.google.com/spreadsheets/d/SHEET_ID/edit)
SHEET_ID = "1HJ9h7UEtUQCXNA58UkZyPsHogJWBAcB1lNWt9nOPMR4"

# Specify the tab name (optional, defaults to the first sheet)
SHEET_NAME = "Trades"

## Option 1:
- Simple way to read a shared google sheet with the view as is

## Option 2:
- Requires google account service credentials to read the full sheet regardless of the filters applied in the browser
- Refer to [`gsheet_access_instructions.txt`](https://share.gemini.google/0dLETC0zvoAe) for step-by-step instructions on how to setup access

In [5]:
gc = gspread.service_account(filename=GSHEET_CREDS)
sh = gc.open_by_key(SHEET_ID)
worksheet = sh.worksheet(SHEET_NAME)
df = pd.DataFrame(worksheet.get_all_records())

print(df.shape)
df.head()

(174, 21)


,Open Time,Open Date,Account,Symbol,Volume,Open Price,Close Price,Commission,Balance at Open,Stop Loss,...,Closed,Risk ($),Potential Profit,PnL,Is Closed,Close Time,Close Date,Entry Link,Exit Link 1,Exit Link 2
0,8/7/2026 11:40:00,8/7/2026,Yvonne's Robinhood,QQQ,-60.00,722.81,722.5,(0.91),,,...,Yes,(0.91),(0.91),17.69,1,8/7/2026 9:15:00,8/7/2026,,,
1,8/7/2026 9:40:00,8/7/2026,Yvonne's Robinhood,QQQ,-50.00,720.68,722.5,(0.75),,,...,Yes,(0.75),(0.75),(91.75),1,8/7/2026 12:26:00,8/7/2026,,,
2,8/9/2026 0:00:00,8/9/2026,FTP - 147367,XAUUSD,-0.13,4358.51,4356.81,(0.91),,,...,Yes,,(0.91),21.19,1,8/9/2026 0:00:00,8/9/2026,,,
3,8/9/2026 0:00:00,8/9/2026,FTP - 147367,US100,-2.40,29868.05,29894.1,0.0,,,...,Yes,,0.0,(62.52),1,8/9/2026 0:00:00,8/9/2026,,,
4,8/9/2026 0:00:00,8/9/2026,FTP - 147367,US100,-1.70,29857.3,29888.1,0.0,,,...,Yes,,0.0,(52.36),1,8/9/2026 0:00:00,8/9/2026,,,


In [6]:
# Keep open trades only
df = df[df.Closed=='No'].copy()

# Fill numberic columns' NA with 0 and cast numeric columns from str to float type
cols = ['Open Price', 'Close Price', 'Commission','Risk ($)', 'Balance at Open', 'PnL']
df[cols] = df[cols].fillna('0')
for c in cols:
    df[c] = df[c].apply(lambda x: float(re.sub(r"\(", "-", re.sub(r"[,\)]", "", str(x))))) # Replace '(' with '-' and remove ')', ',' from the numbers to cast them to float
df.head()

,Open Time,Open Date,Account,Symbol,Volume,Open Price,Close Price,Commission,Balance at Open,Stop Loss,...,Closed,Risk ($),Potential Profit,PnL,Is Closed,Close Time,Close Date,Entry Link,Exit Link 1,Exit Link 2
119,8/24/2026 0:00:00,8/24/2026,Paper Trading #1,COF,-34.00,221.02,216.60,0.00,59800.00,,...,No,-97.92,0.0,150.28,0,,#VALUE!,Link,,
123,8/24/2026 0:00:00,8/24/2026,Tradestation - Equity,CVNA,-140.00,72.32,74.16,-2.91,42336.39,81.13,...,No,-1172.61,3024.49,-259.91,0,,#VALUE!,Link,,
167,8/31/2026 16:01:03,8/31/2026,Paper Trading #2,ETH,-18.34,2480.79,2393.17,0.00,97994.33,2536.9,...,No,-1029.06,2488.19,1606.91,0,,#VALUE!,Link,,
170,9/2/2026 11:24:12,9/2/2026,Tradestation - Equity,COF,-30.00,217.14,216.60,0.00,48759.00,220.61,...,No,-104.10,320.7,16.20,0,,#VALUE!,"Trade 5, 15m",,
172,9/2/2026 11:10:12,9/2/2026,Paper Trading #2,CVNA,-462.00,74.59,74.16,0.00,100331.00,76.86,...,No,-1048.74,2587.2,198.66,0,,#VALUE!,"Trade 4, 15m",,


# Get Point Values

In [7]:
# Specify the tab name (optional, defaults to the first sheet)
SHEET_NAME = 'Symbols'

# Pick an options:
# OPTION 1: Read 'Symbols directly from shared url
# url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME}"
# point_val_df = pd.read_csv(url, header=None, names=['Symbol', 'Point Value'])

# OPTION 2: Read 'Symbols' sheet using using Google authentication - gspread
point_val_df = pd.DataFrame(sh.worksheet(SHEET_NAME).get_all_records(head=0))
all_values = sh.worksheet(SHEET_NAME).get_all_values()
point_val_df = pd.DataFrame(all_values, columns=['Symbol', 'Point Value'])
point_val_df['Point Value'] = point_val_df['Point Value'].astype(float)

point_val_df.head()

,Symbol,Point Value
0,ADA,1.0
1,BTC,1.0
2,COF,1.0
3,CVNA,1.0
4,ETH,1.0


# Get Prices

In [8]:
price_df = pd.DataFrame(df['Symbol']).drop_duplicates()
price_df['Current Price'] = price_df.Symbol.apply(lambda x : get_live_price(x, yfinance_sym_dic))
price_df

,Symbol,Current Price
119,COF,216.600000
123,CVNA,74.160000
167,ETH,2387.159912


# Append Price to trades DF

In [9]:
df = pd.merge(df, price_df, on='Symbol', how='left')
df = pd.merge(df, point_val_df, on='Symbol', how='left')
df['Point Value'] = df['Point Value'].fillna(1)
df['PnL'] = (df['Volume'] * (df['Current Price']-df['Open Price']) * df['Point Value']).round(2)
df

,Open Time,Open Date,Account,Symbol,Volume,Open Price,Close Price,Commission,Balance at Open,Stop Loss,...,Potential Profit,PnL,Is Closed,Close Time,Close Date,Entry Link,Exit Link 1,Exit Link 2,Current Price,Point Value
0,8/24/2026 0:00:00,8/24/2026,Paper Trading #1,COF,-34.00,221.02,216.60,0.00,59800.00,,...,0.0,150.28,0,,#VALUE!,Link,,,216.600000,1.0
1,8/24/2026 0:00:00,8/24/2026,Tradestation - Equity,CVNA,-140.00,72.32,74.16,-2.91,42336.39,81.13,...,3024.49,-257.60,0,,#VALUE!,Link,,,74.160000,1.0
2,8/31/2026 16:01:03,8/31/2026,Paper Trading #2,ETH,-18.34,2480.79,2393.17,0.00,97994.33,2536.9,...,2488.19,1717.18,0,,#VALUE!,Link,,,2387.159912,1.0
3,9/2/2026 11:24:12,9/2/2026,Tradestation - Equity,COF,-30.00,217.14,216.60,0.00,48759.00,220.61,...,320.7,16.20,0,,#VALUE!,"Trade 5, 15m",,,216.600000,1.0
4,9/2/2026 11:10:12,9/2/2026,Paper Trading #2,CVNA,-462.00,74.59,74.16,0.00,100331.00,76.86,...,2587.2,198.66,0,,#VALUE!,"Trade 4, 15m",,,74.160000,1.0
5,9/2/2026 13:35:00,9/2/2026,Tradestation - Equity,CVNA,-25.00,74.81,74.16,0.00,48760.00,76.8,...,170.0,16.25,0,,#VALUE!,,,,74.160000,1.0


# Group by account and symbol to report

In [10]:
spacer_line = '\n\n' + 50 * '-' + '\n'
out = df.groupby('Account').agg({'PnL': sum}).to_string() + spacer_line
out += df.groupby('Symbol').agg({'Volume': sum, 'PnL': sum}).to_string() + spacer_line
out += df.groupby(['Symbol','Account']).agg({'Volume': sum, 'PnL': sum}).to_string() + spacer_line
out += df.groupby(['Account','Symbol']).agg({'Volume': sum, 'PnL': sum}).to_string() + spacer_line
print(out)

                           PnL
Account                       
Paper Trading #1        150.28
Paper Trading #2       1915.84
Tradestation - Equity  -225.15

--------------------------------------------------
        Volume      PnL
Symbol                 
COF     -64.00   166.48
CVNA   -627.00   -42.69
ETH     -18.34  1717.18

--------------------------------------------------
                              Volume      PnL
Symbol Account                               
COF    Paper Trading #1       -34.00   150.28
       Tradestation - Equity  -30.00    16.20
CVNA   Paper Trading #2      -462.00   198.66
       Tradestation - Equity -165.00  -241.35
ETH    Paper Trading #2       -18.34  1717.18

--------------------------------------------------
                              Volume      PnL
Account               Symbol                 
Paper Trading #1      COF     -34.00   150.28
Paper Trading #2      CVNA   -462.00   198.66
                      ETH     -18.34  1717.18
Tradestation - Eq

In [11]:
spacer_line = '\n\n<br>\n\n' 
out = df.groupby('Account').agg({'PnL': sum}).to_markdown() + spacer_line
out += df.groupby('Symbol').agg({'Volume': sum, 'PnL': sum}).to_markdown() + spacer_line
out += df.groupby(['Symbol','Account'], as_index=False).agg({'Volume': sum, 'PnL': sum}).to_markdown() + spacer_line
out += df.groupby(['Account','Symbol'], as_index=False).agg({'Volume': sum, 'PnL': sum}).to_markdown() + spacer_line

print(out)

| Account               |     PnL |
|:----------------------|--------:|
| Paper Trading #1      |  150.28 |
| Paper Trading #2      | 1915.84 |
| Tradestation - Equity | -225.15 |

<br>

| Symbol   |   Volume |     PnL |
|:---------|---------:|--------:|
| COF      |   -64    |  166.48 |
| CVNA     |  -627    |  -42.69 |
| ETH      |   -18.34 | 1717.18 |

<br>

|    | Symbol   | Account               |   Volume |     PnL |
|---:|:---------|:----------------------|---------:|--------:|
|  0 | COF      | Paper Trading #1      |   -34    |  150.28 |
|  1 | COF      | Tradestation - Equity |   -30    |   16.2  |
|  2 | CVNA     | Paper Trading #2      |  -462    |  198.66 |
|  3 | CVNA     | Tradestation - Equity |  -165    | -241.35 |
|  4 | ETH      | Paper Trading #2      |   -18.34 | 1717.18 |

<br>

|    | Account               | Symbol   |   Volume |     PnL |
|---:|:----------------------|:---------|---------:|--------:|
|  0 | Paper Trading #1      | COF      |   -34    |  150.28 |

In [12]:
if GENERATE_MARKDOWN:
    page_nm = 'acct_lvl_stats.md'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:  # Save to a file
        f.write(df.groupby('Account').agg({'PnL': sum}).to_markdown())
        
    page_nm = 'sym_lvl_stats.md'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        f.write(df.groupby('Symbol').agg({'Volume': sum, 'PnL': sum}).to_markdown())
    
    page_nm = 'sym_acct_lvl_stats.md'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        f.write(df.groupby(['Symbol','Account'], as_index=False).agg({'Volume': sum, 'PnL': sum}).to_markdown())
    
    page_nm = 'acct_sym_lvl_stats.md'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        f.write(df.groupby(['Account','Symbol'], as_index=False).agg({'Volume': sum, 'PnL': sum}).to_markdown())

In [13]:
if GENERATE_HTML:
    page_nm = 'acct_lvl_stats.html'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:  # Save to a file
        t = df.groupby('Account').agg({'PnL': sum})
        f.write(t.to_html(border=0, justify='left',  table_id='dataTable', classes='table table-striped table-hover'))
        
    page_nm = 'sym_lvl_stats.html'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        t = df.groupby('Symbol').agg({'Volume': sum, 'PnL': sum})
        f.write(t.to_html(border=0, justify='left',  table_id='dataTable', classes='table table-striped table-hover'))
    
    page_nm = 'sym_acct_lvl_stats.html'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        t = df.groupby(['Symbol','Account']).agg({'Volume': sum, 'PnL': sum})
        f.write(t.to_html(border=0, justify='left',  table_id='dataTable', classes='table table-striped table-hover'))
    
    page_nm = 'acct_sym_lvl_stats.html'
    with open(os.path.join(REPORT_PATH, page_nm), 'w') as f:
        t = df.groupby(['Account','Symbol']).agg({'Volume': sum, 'PnL': sum})
        f.write(t.to_html(border=0, justify='left',  table_id='dataTable', classes='table table-striped table-hover'))